### Error Distribution Calculation for Learner's corpus

In [4]:
import pandas as pd
from difflib import SequenceMatcher
from collections import Counter

def count_error_types_for_pair(src_sent, tgt_sent):
    """
    Compare Source_Text (learner) and Target_Text (corrected) at token level
    and count:
      - substitution errors
      - deletion (omission) errors
      - insertion (addition) errors
    according to our learner-perspective mapping.
    """
    src_tokens = str(src_sent).split()
    tgt_tokens = str(tgt_sent).split()

    sm = SequenceMatcher(None, src_tokens, tgt_tokens)

    subs = dels = ins = 0  # substitution, deletion, insertion

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue

        elif tag == "replace":
            # tokens in src[i1:i2] are replaced by tgt[j1:j2]
            # count this as ONE substitution error (like an R edit)
            subs += 1

        elif tag == "insert":
            # tokens tgt[j1:j2] inserted in the correction
            # => learner omitted them  => deletion/omission error
            dels += 1

        elif tag == "delete":
            # tokens src[i1:i2] deleted in the correction
            # => learner added extra tokens => insertion/addition error
            ins += 1

    return subs, dels, ins


def compute_error_distribution_from_final(final_csv_path):
    """
    Read FINAL_seg_merged_with_labeled.csv and compute global counts + ratios
    of substitution / deletion / insertion errors for the SOURCE text,
    using the M2-style logic.
    """
    df = pd.read_csv(final_csv_path)

    total_sub = total_del = total_ins = 0

    for _, row in df.iterrows():
        src = row["Source_Text"]
        tgt = row["Target_Text"]
        s, d, i = count_error_types_for_pair(src, tgt)
        total_sub += s
        total_del += d
        total_ins += i

    total_errors = total_sub + total_del + total_ins
    if total_errors == 0:
        raise ValueError("No differences found between Source_Text and Target_Text.")

    learner_counts = {
        "substitution": total_sub,
        "deletion": total_del,    # learner omitted tokens
        "insertion": total_ins    # learner added extra tokens
    }

    learner_ratios = {
        k: v / total_errors for k, v in learner_counts.items()
    }

    return {
        "learner_counts": learner_counts,
        "learner_ratios": learner_ratios,
        "total_errors": total_errors
    }


# ===== Example usage =====
if __name__ == "__main__":
    final_csv_path = "FINAL_seg_merged_with_labeled.csv"
    stats = compute_error_distribution_from_final(final_csv_path)

    print("=== Learner-perspective counts (from FINAL source text) ===")
    for k, v in stats["learner_counts"].items():
        print(f"{k:12s}: {v}")

    print("\n=== Learner-perspective ratios (from FINAL source text) ===")
    for k, v in stats["learner_ratios"].items():
        print(f"{k:12s}: {v:.3f}")

    print(f"\nTotal edit operations (all types): {stats['total_errors']}")


=== Learner-perspective counts (from FINAL source text) ===
substitution: 1750
deletion    : 1150
insertion   : 721

=== Learner-perspective ratios (from FINAL source text) ===
substitution: 0.483
deletion    : 0.318
insertion   : 0.199

Total edit operations (all types): 3621


In [5]:
import pandas as pd
from difflib import SequenceMatcher

def count_error_types_for_pair(src_sent, tgt_sent):
    """
    Compare Source_Text (learner) and Target_Text (corrected) at token level
    and count:
      - substitution errors
      - deletion (omission) errors
      - insertion (addition) errors
    according to learner-perspective mapping:

      replace  -> substitution
      insert   -> deletion (learner omitted tokens)
      delete   -> insertion (learner added extra tokens)
    """
    src_tokens = str(src_sent).split()
    tgt_tokens = str(tgt_sent).split()

    sm = SequenceMatcher(None, src_tokens, tgt_tokens)

    subs = dels = ins = 0  # substitution, deletion, insertion

    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag == "equal":
            continue

        elif tag == "replace":
            # src[i1:i2] replaced by tgt[j1:j2]
            # treat as ONE substitution error (like an R edit)
            subs += 1

        elif tag == "insert":
            # tgt[j1:j2] inserted in correction
            # => learner omitted these tokens => deletion error
            dels += 1

        elif tag == "delete":
            # src[i1:i2] deleted in correction
            # => learner added extra tokens => insertion error
            ins += 1

    return subs, dels, ins


def add_error_counts_to_final(final_csv_path,
                              output_csv_path="FINAL_seg_merged_with_labeled_with_error_counts.csv"):
    """
    Read FINAL_seg_merged_with_labeled.csv, compute per-sentence
    substitution / deletion / insertion counts, and save a new CSV
    with columns: num_sub, num_del, num_ins.
    """
    df = pd.read_csv(final_csv_path)

    num_sub_list = []
    num_del_list = []
    num_ins_list = []

    for _, row in df.iterrows():
        src = row["Source_Text"]
        tgt = row["Target_Text"]
        s, d, i = count_error_types_for_pair(src, tgt)
        num_sub_list.append(s)
        num_del_list.append(d)
        num_ins_list.append(i)

    df["num_sub"] = num_sub_list
    df["num_del"] = num_del_list
    df["num_ins"] = num_ins_list

    df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
    print(f"Saved new file with error counts to: {output_csv_path}")


# ===== Example usage =====
if __name__ == "__main__":
    input_path = "FINAL_seg_merged_with_labeled.csv"
    output_path = "FINAL_seg_merged_with_labeled_with_error_counts.csv"
    add_error_counts_to_final(input_path, output_path)


Saved new file with error counts to: FINAL_seg_merged_with_labeled_with_error_counts.csv
